# Getting Started with molfp

**molfp** is a Python library for computing, interpreting, and analyzing molecular fingerprints. It provides 13 fingerprint types built on native RDKit (no Java dependency), plus tools for explainability, SHAP interpretation, collision detection, activity cliff analysis, and more.

```
pip install molfp
```

## 1. Computing Fingerprints

### 1.1 Single Molecule
Pass a SMILES string (or RDKit Mol), get back a list of integers.

In [ ]:
import molfp

aspirin = "CC(=O)Oc1ccccc1C(=O)O"

fp = molfp.maccs(aspirin)
print(f"MACCS:   {len(fp)} bits, {sum(fp)} ON")

fp = molfp.ecfp4(aspirin)
print(f"ECFP4:   {len(fp)} bits, {sum(fp)} ON")

fp = molfp.pubchem(aspirin)
print(f"PubChem: {len(fp)} bits, {sum(fp)} ON")

### 1.2 All 13 Fingerprint Types

In [ ]:
for name in molfp.available():
    fn, n_bits, desc = molfp.get(name)
    fp = fn(aspirin)
    print(f"  {name:>22s}  {n_bits:>5d} bits  {sum(fp):>4d} ON  ({desc})")

### 1.3 Batch Computation
Batch functions return numpy arrays. Use `n_jobs=-1` for parallelism and `sparse=True` for memory efficiency.

In [ ]:
smiles = ["CCO", "c1ccccc1", aspirin, "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"]

# Dense
fps = molfp.maccs_batch(smiles)
print(f"Dense:  shape={fps.shape}, dtype={fps.dtype}")

# Sparse (97% memory savings for Klekota-Roth)
fps_sp = molfp.klekota_roth_batch(smiles, sparse=True, n_jobs=-1)
print(f"Sparse: shape={fps_sp.shape}, nnz={fps_sp.nnz}, density={fps_sp.nnz/(fps_sp.shape[0]*fps_sp.shape[1]):.2%}")

# Universal API
fps = molfp.compute_batch(smiles, "ecfp4", n_jobs=-1)
print(f"compute_batch: shape={fps.shape}")

## 2. Structure Standardization
Desalt, neutralize, canonicalize tautomers, and deduplicate by InChIKey — critical before FP computation.

In [ ]:
cases = [
    ("CC(=O)Oc1ccccc1C(=O)[O-].[Na+]", "sodium aspirin"),
    ("O=C1CC=CC=C1",                    "cyclohexadienone"),
    ("c1ccc(O)c(C(=O)O)c1",            "salicylic acid (alt SMILES)"),
]
for smi, desc in cases:
    print(f"  {desc:>30s}: {smi:>40s}  ->  {molfp.standardize_smiles(smi)}")

In [ ]:
# Full preprocessing pipeline
raw = ["CC(=O)Oc1ccccc1C(=O)O", "CC(=O)Oc1ccccc1C(=O)[O-].[Na+]", "INVALID", "CCO", "OCC"]
result = molfp.preprocess(raw)
print(f"Input: {result[chr(39)+'n_input'+chr(39)]} -> Standardized: {result[chr(39)+'n_after_standardization'+chr(39)]} -> Unique: {result[chr(39)+'n_after_deduplication'+chr(39)]}")
print(f"Clean: {result[chr(39)+'clean_smiles'+chr(39)]}")

## 3. Bit-Level Explainability
### 3.1 explain() — Which atoms set which bits?

In [ ]:
# ECFP4: each bit -> (center_atom, radius) environments
exp = molfp.explain(aspirin, "ecfp4")
print(f"ECFP4: {len(exp)} ON bits explained")
for bit, envs in list(exp.items())[:4]:
    print(f"  bit {bit}: {envs}")

### 3.2 annotate() — Chemical names for every bit

In [ ]:
for bit in [0, 83, 84, 274]:
    ann = molfp.annotate("substructure", bit)
    print(f"  bit {bit:>3d}: {ann[chr(39)+'name'+chr(39)]:>30s}  SMARTS: {ann.get(chr(39)+'smarts'+chr(39),chr(39)+chr(39))[:35]}")

print()
for bit in [9, 178, 255]:
    ann = molfp.annotate("pubchem", bit)
    print(f"  bit {bit:>3d}: [{ann[chr(39)+'category'+chr(39)]:>15s}] {ann[chr(39)+'description'+chr(39)]}")

## 4. Distance & Similarity
molfp provides 5 vectorized metrics: Tanimoto, Dice, Cosine, Sokal-Michener, Russel-Rao.

In [ ]:
import numpy as np
fp_a, fp_b = np.array(molfp.maccs("CCO")), np.array(molfp.maccs(aspirin))
print(f"Tanimoto: {molfp.tanimoto(fp_a, fp_b):.3f}")
print(f"Dice:     {molfp.dice(fp_a, fp_b):.3f}")
print(f"Cosine:   {molfp.cosine(fp_a, fp_b):.3f}")

# Vectorized pairwise matrix
smiles5 = ["CCO", "c1ccccc1", aspirin, "OC(=O)c1ccccc1O", "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"]
fps = molfp.maccs_batch(smiles5)
mat = molfp.pairwise(fps, "tanimoto")
print(f"
Pairwise matrix: shape={mat.shape}")

## 5. Hash Collision Detection
Detect when different substructures hash to the same bit — a known problem that corrupts SHAP values.

In [ ]:
for size in [64, 256, 1024, 4096]:
    coll = molfp.detect_collisions("c1ccc2c(c1)cc1ccc3ccccc3c1c2", "ecfp4", size)
    n_on = sum(molfp.ecfp4("c1ccc2c(c1)cc1ccc3ccccc3c1c2", n_bits=size))
    print(f"  {size:>5d} bits: {len(coll):>2d} collisions / {n_on:>3d} ON bits")

## 6. Bioactivity Profiling
Classify molecules by activity, analyze intra/inter-class similarity, find enriched substructures.

In [ ]:
import csv
with open("data/processed/aromatase_bioactivity_curated.csv", newline="") as f:
    rows = list(csv.DictReader(f))
all_smi = [r["canonical_smiles"] for r in rows if r["pchembl_value"]]
all_act = [float(r["pchembl_value"]) for r in rows if r["pchembl_value"]]
np.random.seed(42); idx = np.random.choice(len(all_smi), 200, replace=False)
smi, act = [all_smi[i] for i in idx], [all_act[i] for i in idx]

labels = molfp.classify(act, metric="pchembl")
counts = {}
for l in labels: counts[l] = counts.get(l, 0) + 1
print(f"Classes: {counts}")

enriched = molfp.enriched_bits(smi, labels, "maccs", min_enrichment=1.5)
for cls, bits in sorted(enriched.items()):
    print(f"
{cls}: {len(bits)} enriched bits")
    for bit, cf, of, ratio in bits[:3]:
        ann = molfp.annotate("maccs", bit)
        print(f"  bit {bit:>3d}: class={cf:.0%} vs overall={of:.0%} -> {ratio:.1f}x")

## 7. Activity Cliffs
Find structurally similar molecules with large activity differences.

In [ ]:
cliffs = molfp.activity_cliffs(smi, act, "maccs", sim_threshold=0.5, activity_threshold=2.0)
print(f"{len(cliffs)} cliffs found")
for c in cliffs[:3]:
    print(f"  sim={c[chr(39)+'similarity'+chr(39)]:.2f}  delta={c[chr(39)+'delta_activity'+chr(39)]:.1f}")

## 8. Consensus / Fusion Fingerprints
Select the most informative bits from multiple FP types.

In [ ]:
result = molfp.fuse(smi, act, fp_types=["maccs", "estate", "pubchem"], n_bits=20, method="variance")
sources = {}
for col, (fp_type, bit) in result["source_map"].items():
    sources[fp_type] = sources.get(fp_type, 0) + 1
print(f"Fused FP shape: {result[chr(39)+'fused_fps'+chr(39)].shape}")
print(f"Bits per FP type: {sources}")

## 9. Isomer Discrimination
Test how well each fingerprint distinguishes different molecules.

In [ ]:
dp = molfp.discriminative_power(smi[:50], ["maccs", "ecfp4", "pubchem", "rdk"])
for r in dp:
    print(f"  {r[chr(39)+'fp_type'+chr(39)]:>10s}: {r[chr(39)+'unique_fps'+chr(39)]}/{r[chr(39)+'total_molecules'+chr(39)]} unique ({r[chr(39)+'discrimination_ratio'+chr(39)]:.3f})")

## 10. SHAP Interpretation with Chemical Meaning
The flagship feature: train a model, compute SHAP values, and map each important bit to its chemical meaning with present/absent effect.

In [ ]:
result = molfp.interpret(smi, act, fp_type="maccs", metric="pchembl")
print(f"Model accuracy: {result[chr(39)+'accuracy'+chr(39)]:.1%}")

print("
Top 5 important bits:")
for item in result["feature_importance"][:5]:
    print(f"  bit {item[chr(39)+'bit'+chr(39)]:>3d}  SHAP={item[chr(39)+'mean_abs_shap'+chr(39)]:.4f}  {item[chr(39)+'chemical_meaning'+chr(39)]}")

In [ ]:
# Per-class drivers with present/absent effect
for cls in result["label_set"]:
    drivers = result["class_drivers"][cls]
    pro = [d for d in drivers if d["direction"] == "promotes"][:3]
    inh = [d for d in drivers if d["direction"] == "inhibits"][:2]
    print(f"
{cls.upper()}:")
    for d in pro:
        print(f"  ↑ {d[chr(39)+'value_effect'+chr(39)]:>7s}  {d[chr(39)+'chemical_meaning'+chr(39)]:>25s}  SHAP {d[chr(39)+'mean_shap'+chr(39)]:+.4f}  (freq {d[chr(39)+'freq_in_class'+chr(39)]:.0%} vs {d[chr(39)+'freq_overall'+chr(39)]:.0%})")
    for d in inh:
        print(f"  ↓ {d[chr(39)+'value_effect'+chr(39)]:>7s}  {d[chr(39)+'chemical_meaning'+chr(39)]:>25s}  SHAP {d[chr(39)+'mean_shap'+chr(39)]:+.4f}  (freq {d[chr(39)+'freq_in_class'+chr(39)]:.0%} vs {d[chr(39)+'freq_overall'+chr(39)]:.0%})")

## 11. scikit-learn Integration
All 13 FP types as sklearn transformers — works with Pipeline, GridSearchCV, FeatureUnion.

In [ ]:
from sklearn.pipeline import make_pipeline, make_union
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV

y = [1 if l == "active" else 0 for l in labels]

# Pipeline: FP -> classifier
pipe = make_pipeline(molfp.ECFPTransformer(n_bits=1024), RandomForestClassifier(n_estimators=50, random_state=42))
scores = cross_val_score(pipe, smi, y, cv=3, scoring="accuracy")
print(f"ECFP4 -> RF: {scores.mean():.3f} +/- {scores.std():.3f}")

# Feature union: combine FP types
union_pipe = make_pipeline(
    make_union(molfp.ECFPTransformer(n_bits=512), molfp.MACCSTransformer(), molfp.EStateTransformer()),
    RandomForestClassifier(n_estimators=50, random_state=42)
)
scores = cross_val_score(union_pipe, smi, y, cv=3, scoring="accuracy")
print(f"ECFP+MACCS+EState -> RF: {scores.mean():.3f} +/- {scores.std():.3f}")

## Summary

| Module | Key Functions |
|--------|---------------|
| **Fingerprints** | `maccs()`, `ecfp4()`, `pubchem()`, + 10 more |
| **Standardize** | `preprocess()`, `deduplicate()`, `standardize_smiles()` |
| **Explain** | `explain()` — bit -> atom mapping for all 13 FP types |
| **Annotate** | `annotate()` — chemical names for every bit |
| **Interpret** | `interpret()` — SHAP + chemical meaning + present/absent |
| **Distance** | `tanimoto()`, `pairwise()` — 5 vectorized metrics |
| **Collisions** | `detect_collisions()`, `collision_rate()` |
| **Bioactivity** | `classify()`, `enriched_bits()`, `class_similarity()` |
| **Cliffs** | `activity_cliffs()`, `cliff_summary()` |
| **Fusion** | `fuse()`, `complementarity()` |
| **Isomers** | `discriminative_power()`, `isomer_sensitivity()` |
| **sklearn** | 13 transformer classes for Pipeline/GridSearchCV |